# Sequential Recommender with GRU (GRU4Rec)
This notebook implements a GRU-based sequential recommender using user interaction sequences.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


## 1. Load and Prepare Data
We load the interactions dataset and group by user to create sequences of item interactions sorted by time.

In [ ]:
# Load dataset
df = pd.read_csv('dataset/archive_3/interactions_train.csv')

# Ensure date is datetime and sort
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['user_id', 'date'])

# Re-map item IDs to start from 1 (0 will be used for padding)
item_mapping = {item: idx + 1 for idx, item in enumerate(df['recipe_id'].unique())}
df['item_idx'] = df['recipe_id'].map(item_mapping)
num_items = len(item_mapping) + 1 # +1 for padding index 0
print(f"Total unique items: {num_items - 1}")

# Group by user to form sequences
user_seqs = df.groupby('user_id')['item_idx'].apply(list).reset_index()

# Keep sequences with at least 3 items
user_seqs = user_seqs[user_seqs['item_idx'].apply(len) >= 3]
print(f"Total users with sequences >= 3: {len(user_seqs)}")


## 2. PyTorch Dataset
Create a dataset that generates input sequences of fixed length and target items.

In [ ]:
class GRUDataset(Dataset):
    def __init__(self, sequences, max_len=10):
        self.inputs = []
        self.targets = []
        
        for seq in sequences:
            # Create sub-sequences
            for i in range(2, len(seq)):
                input_seq = seq[max(0, i - max_len):i]
                target = seq[i]
                
                # Pad sequence if it's shorter than max_len
                if len(input_seq) < max_len:
                    input_seq = [0] * (max_len - len(input_seq)) + input_seq
                    
                self.inputs.append(input_seq)
                self.targets.append(target)
                
    def __len__(self):
        return len(self.inputs)
        
    def __getitem__(self, idx):
        return torch.tensor(self.inputs[idx], dtype=torch.long), torch.tensor(self.targets[idx], dtype=torch.long)

# Instantiate dataset and dataloader
# Limit to a subset for faster training demonstration
max_seq_len = 10
train_dataset = GRUDataset(user_seqs['item_idx'].values[:5000], max_len=max_seq_len)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
print(f"Total training samples: {len(train_dataset)}")


## 3. Define the GRU4Rec Model
A simple implementation of GRU for sequential recommendation.

In [ ]:
class GRU4Rec(nn.Module):
    def __init__(self, num_items, embedding_dim=64, hidden_dim=128, num_layers=1, dropout=0.2):
        super(GRU4Rec, self).__init__()
        self.item_embedding = nn.Embedding(num_items, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(
            input_size=embedding_dim, 
            hidden_size=hidden_dim, 
            num_layers=num_layers, 
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_items)
        
    def forward(self, x):
        # x shape: (batch_size, seq_len)
        embedded = self.item_embedding(x)
        # embedded shape: (batch_size, seq_len, embedding_dim)
        
        gru_out, hidden = self.gru(embedded)
        # We only need the output of the last step
        last_out = gru_out[:, -1, :]
        last_out = self.dropout(last_out)
        
        out = self.fc(last_out)
        return out

model = GRU4Rec(num_items=num_items, embedding_dim=64, hidden_dim=128).to(device)
print(model)


## 4. Training Loop
Train the model using CrossEntropyLoss.

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 5

model.train()
for epoch in range(epochs):
    total_loss = 0
    for inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")


## 5. Inference / Recommendation
Generate recommendations for a sample user sequence.

In [ ]:
model.eval()

# Sample a sequence from the dataset
sample_seq = user_seqs['item_idx'].iloc[0][:5]
print("Input sequence (item IDs):", sample_seq)

with torch.no_grad():
    # Pad to max_len
    input_tensor = sample_seq.copy()
    if len(input_tensor) < max_seq_len:
        input_tensor = [0] * (max_seq_len - len(input_tensor)) + input_tensor
    
    input_tensor = torch.tensor([input_tensor], dtype=torch.long).to(device)
    output = model(input_tensor)
    
    # Get top 5 recommendations
    top_k = 5
    _, top_indices = torch.topk(output[0], top_k)
    
    print(f"Top {top_k} recommended item IDs:")
    for idx in top_indices:
        print(idx.item())
